# Can ML Prioritize Content Refresh Reviews?

**Lane:** Refresh / Content Opportunity Scoring  
**Unit of analysis:** one pseudonymized content item  
**Intended use:** human-reviewed prioritization only

This notebook is the reproducibility companion to the deployed research paper. The full modeling, baseline, validation, and action logic remains visible in the Week 4–7 notebooks; this companion reads their committed evidence receipts and checks that the final story is internally consistent.

## Abstract

This study asks whether measured search and content signals can prioritize which pages a human editor reviews first. It models 30,000 pseudonymized FlyRank starter-release pages and checks the design on a 71,777-page March 2026 frame built from the 78,835,655-row warehouse release. A transparent rule, Logistic Regression, and Random Forest are compared on the same client-grouped starter holdout using five predeclared features, while the warehouse check separates March 1–21 features from a March 22–31 outcome. Logistic Regression reached 75% Precision@20 but 0.501 AUROC, Random Forest reached 0.625 AUROC and 0.617 average precision, and the stricter warehouse check reached 0.590 AUROC. The result is therefore a ranked, human-reviewed decision-support playbook—not evidence that refreshing a page causes recovery.

## 1. Introduction / Problem statement

Content teams have more pages than they can inspect one by one. The supported decision is **which pages an editor should review first when time is limited**. The output is a score, a ranked queue, and a reason code; it is not an automatic editing instruction. False positives waste editorial time, while false negatives leave possible opportunities unreviewed.

## 2. Data

- **Primary model:** real, anonymized 30,000-row × 44-column starter release; 32 pseudonymized clients; one row per content item.
- **Warehouse support:** FlyRank internship-warehouse build `v20260703`, table `fact_content_daily_performance`; 78,835,655 daily rows from 2025-01-27 through 2026-06-30.
- **March contract:** 9,841,378 daily rows and 331,437 content items; features from 2026-03-01 through 2026-03-21; outcome from 2026-03-22 through 2026-03-31; 71,777 eligible pages after requiring at least 100 impressions in the preceding ten days.
- **Excluded:** outcome-window measurements, label-derived fields, IDs as predictors, client names, domains, URLs, private queries, credentials, and product decisions. June 2026 remains sealed during label development.

In [1]:
from pathlib import Path
from urllib.request import urlopen
import json
import re
import pandas as pd

REPO = "aabdullahhtar-create/flyrank-ML-internship"
RAW_ROOT = f"https://raw.githubusercontent.com/{REPO}/main"

root_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (p for p in root_candidates if (p / "work/Outputs/capstone_metrics.json").exists()),
    None,
)

def read_text(relative_path):
    if ROOT is not None:
        return (ROOT / relative_path).read_text(encoding="utf-8")
    with urlopen(f"{RAW_ROOT}/{relative_path}", timeout=30) as response:
        return response.read().decode("utf-8")

def read_json(relative_path):
    return json.loads(read_text(relative_path))

metrics = read_json("work/Outputs/capstone_metrics.json")
playbook = read_json("work/Outputs/w07_playbook_metrics.json")
data_contract = read_json("work/notebooks/w03_data_contract.ipynb")

paper_lines = read_text("submission/paper_url.txt").splitlines()
assert len(paper_lines) == 1 and paper_lines[0].startswith("https://")
PAPER_URL = paper_lines[0]

print("Evidence source:", "local repository" if ROOT else "public GitHub repository")
print("Deployed paper:", PAPER_URL)

Evidence source: local repository
Deployed paper: https://aabdullahhtar-create.github.io/flyrank-ML-internship/


## 3. Methodology

The transparent rule baseline ranks pages with meaningful impressions, positions 1–20, and a low-CTR gap. Logistic Regression is the readable first model; Random Forest is the nonlinear challenger. Both use five predeclared inputs: impressions, clicks, CTR, average position, and days since last update.

The primary split is `GroupShuffleSplit(test_size=0.25, random_state=42)`: 24 clients / 22,885 pages for training and 8 unseen clients / 7,115 pages for testing, with zero client overlap. Precision@20 is primary because the output is a capacity-limited queue; Precision@50, AUROC, average precision, and the held-out base rate are reported beside it. `trend_direction`, `trend_pct`, the proxy itself, and IDs are never model features.

In [2]:
results = pd.DataFrame(metrics["results"])
expected_methods = {
    "Base rate",
    "Week-4 baseline",
    "Logistic Regression",
    "Random Forest",
}
assert set(results["Method"]) == expected_methods
assert metrics["grouped_test_base_rate"] == 0.517
assert "zero overlap" in metrics["validation"]

formatted_results = results.copy()
for column in ["Precision@20", "Precision@50"]:
    formatted_results[column] = formatted_results[column].map(lambda value: f"{value:.1%}")
for column in ["AUROC", "Average Precision"]:
    formatted_results[column] = formatted_results[column].map(lambda value: f"{value:.3f}")
display(formatted_results)
print(metrics["validation"])
print(metrics["claim"])

Grouped by client; 24 train clients, 8 test clients; zero overlap; random_state=42
Exploratory human-reviewed decision-support; no causal refresh claim.


             Method Precision@20 Precision@50 AUROC Average Precision
          Base rate        51.7%        51.7% 0.500             0.517
    Week-4 baseline        50.0%        58.0% 0.500             0.513
Logistic Regression        75.0%        62.0% 0.501             0.525
      Random Forest        60.0%        72.0% 0.625             0.617

In [3]:
contract_blob = json.dumps(data_contract)
warehouse_checks = {
    "March daily rows": "9,841,378" if "9841378" in contract_blob else None,
    "March content items": "331,437" if "331437" in contract_blob else None,
    "Eligible feature-frame pages": "71,777" if "Feature-frame rows: 71,777" in contract_blob else None,
    "Forward-decline proxy rate": "23.4%" if "Observed decline-proxy rate: 23.4%" in contract_blob else None,
    "Honest grouped AUROC": "0.590" if "Honest five-feature held-out AUROC: 0.590" in contract_blob else None,
    "Deliberate leaked AUROC": "1.000 (leak removed)" if "With deliberate label leak AUROC: 1.000" in contract_blob else None,
}
assert all(value is not None for value in warehouse_checks.values())
display(pd.DataFrame(warehouse_checks.items(), columns=["Warehouse check", "Committed result"]))

             Warehouse check     Committed result
            March daily rows            9,841,378
         March content items              331,437
Eligible feature-frame pages               71,777
  Forward-decline proxy rate                23.4%
        Honest grouped AUROC                0.590
     Deliberate leaked AUROC 1.000 (leak removed)

## 4. Results

The grouped starter test base rate is 51.7%. Logistic Regression leads Precision@20 at 75%, but its AUROC is 0.501; that small top-k win does not establish reliable overall discrimination. Random Forest leads broader discrimination at AUROC 0.625 and average precision 0.617, and reaches 72% Precision@50.

The warehouse support check is deliberately separate because it uses a future-window label: its client-grouped five-feature Logistic Regression reaches AUROC 0.590. The deliberate label-copy experiment reaches 1.000, demonstrating the leakage trap, and the copied field is removed before retaining the honest result.

## 5. Limitations & honest framing

- The starter release is a trailing-90-day snapshot, not an intervention study.
- The warehouse ten-day impression outcome can reflect seasonality, demand, SERP changes, or measurement noise.
- CTR depends on ranking and result-page layout; staleness does not imply poor quality.
- Precision@20 covers only twenty held-out cases and is not sufficient alone.
- Grouped validation tests unseen clients, but the primary starter result is not a future-month test.
- No result proves that refreshing, rewriting, pruning, merging, or changing metadata causes recovery.

**Claim boundary:** observed and directional decision support only.

## 6. Ranked recommendations

1. **REVIEW_TITLE_SNIPPET / LOW_CTR_VISIBLE** — inspect intent, position, SERP features, and snippet accuracy before changing metadata.
2. **REVIEW_REFRESH / STALE_HIGH_VOLUME** — inspect factual freshness, traffic history, cannibalization, and recent changes before editing.
3. **MONITOR / VISIBLE_MONITOR** — preserve the page and collect a future window rather than creating churn.
4. **HOLD / INSUFFICIENT_SIGNAL** — gather more evidence or leave unchanged.

No automatic publishing, rewriting, deletion, noindexing, redirecting, canonicalization, merging, metadata changes, or traffic promises.

In [4]:
action_counts = pd.Series(playbook["action_counts"], name="pages").rename_axis("action")
reason_counts = pd.Series(playbook["reason_counts"], name="pages").rename_axis("reason_code")

assert int(action_counts.sum()) == playbook["rows_ranked"] == 30_000
assert int(reason_counts.sum()) == 30_000

display(action_counts.sort_values(ascending=False).to_frame())
display(reason_counts.sort_values(ascending=False).to_frame())
print("Intended use:", playbook["monitoring"]["intended_use"])

Intended use: Human-reviewed prioritization only


 pages
 17971
  9759
  2264
     6

 pages
 17971
  9759
  2264
     6

## 7. Reproducibility

- [Public repository](https://github.com/aabdullahhtar-create/flyrank-ML-internship)
- [Warehouse data contract](https://github.com/aabdullahhtar-create/flyrank-ML-internship/blob/main/work/notebooks/w03_data_contract.ipynb)
- [Transparent baseline](https://github.com/aabdullahhtar-create/flyrank-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)
- [Model comparison](https://github.com/aabdullahhtar-create/flyrank-ML-internship/blob/main/work/notebooks/w05_model.ipynb)
- [Validation audit](https://github.com/aabdullahhtar-create/flyrank-ML-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)
- [Ranked action playbook](https://github.com/aabdullahhtar-create/flyrank-ML-internship/blob/main/work/notebooks/w07_action_playbook.ipynb)
- [Deployed paper](https://aabdullahhtar-create.github.io/flyrank-ML-internship/)

Random seed is fixed at 42 where applicable. The committed notebooks contain the executable code and output evidence; this companion validates their receipts without exposing private data.

## 8. Acknowledgments & data credit

[**Built on the FlyRank ML Internship dataset**](https://flyrank.ai/). Thanks to FlyRank for the pseudonymized internship data and learning framework.

## ML-12: five-minute demo outline

- **0:00–0:40 — Decision:** editors need a small, ordered review queue, not another undifferentiated dashboard.
- **0:40–1:25 — Data:** explain the 30,000-page starter model and the separate March warehouse future-window check.
- **1:25–2:15 — Method:** show the transparent rule, five features, two models, grouped-client split, and leakage exclusions.
- **2:15–3:15 — Result:** contrast 75% Precision@20 with 0.501 Logistic Regression AUROC, then show Random Forest AUROC 0.625 and the warehouse check at 0.590.
- **3:15–4:20 — Action:** walk through REVIEW_TITLE_SNIPPET, REVIEW_REFRESH, MONITOR, and HOLD with reason codes.
- **4:20–5:00 — Boundary:** observational decision support only; no automatic edits and no causal refresh claim.

## Social-post cut

I built a public-safe content-review prioritization study on real FlyRank search data. A Logistic Regression put 75% observed declining pages in its top 20 on unseen clients, but its 0.501 AUROC showed why one strong top-k result is not enough; Random Forest reached 0.625 AUROC, while a stricter warehouse future-window check reached 0.590. The final output is a ranked, reason-coded queue for human review—not an auto-edit system or a causal promise.

## Employer-facing three-sentence summary

I built a reproducible content-opportunity scoring workflow that turns measurable search signals into a ranked, reason-coded review queue. I validated a rule baseline, Logistic Regression, and Random Forest on 30,000 pseudonymized pages with whole clients held out, and I checked the design against a 71,777-page future-window frame built from a 78.8-million-row warehouse. I reported mixed results honestly and constrained the system to human decision support rather than automatic content changes.

In [5]:
required_sections = [
    "Abstract",
    "Introduction / Problem statement",
    "Data",
    "Methodology",
    "Results",
    "Limitations & honest framing",
    "Ranked recommendations",
    "Reproducibility",
    "Acknowledgments & data credit",
]
capstone_document = read_json("work/notebooks/capstone.ipynb")
notebook_text = "\n".join(
    "".join(cell.get("source", []))
    for cell in capstone_document.get("cells", [])
)

assert len(results) == 4
assert int(action_counts.sum()) == 30_000
assert PAPER_URL == "https://aabdullahhtar-create.github.io/flyrank-ML-internship/"
assert all(warehouse_checks.values())
assert all(section in notebook_text for section in required_sections)

print("Capstone receipt checks passed.")
print("Paper URL is one clean line; model, warehouse, and playbook evidence loaded successfully.")

Capstone receipt checks passed.
Paper URL is one clean line; model, warehouse, and playbook evidence loaded successfully.
